# Paper — 01c: Canny Edge Orientation

Tests two Canny-based orientation estimators and compares them against the baseline and full structure tensor from `paper_01b`.

**Methods compared:**
1. Baseline: segmentize → EllipseModel → MRR (current pipeline)
2. Moments: polygon area PCA (no raster needed)
3. Canny PCA: Canny on image → PCA on edge pixel *locations*
4. Canny tensor: Canny on image → structure tensor at edge pixels only
5. Full tensor: structure tensor on *all* interior pixels (Fix B from 01b)

**Experiments:**
- A. All 5 methods × 3 size bins
- B. Canny σ sensitivity
- C. KS uniformity statistics

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
import rasterio
from rasterio.mask import mask as rio_mask
from skimage.feature import canny as skimage_canny
from scipy.ndimage import sobel as nd_sobel
from scipy.stats import kstest
from tqdm import tqdm

from rastertools_BOULDERING import metadata as raster_metadata
from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row
from shapely import segmentize

In [ ]:
work_dir  = Path.home() / "tmp" / "YOLOv8BeyondEarth"
in_raster = Path("/scratch/users/cayleigh/test_raster/M1221383405.tif")

PRED_CONFIGS = {
    "YOLOv8":          (work_dir / "exp_yolo_256",           "*-downscaled-mask-nms.shp"),
    "SAM2 zero-shot":  (work_dir / "exp_sam2_256",           "*-downscaled-mask-nms.shp"),
    "SAM2 fine-tuned": (work_dir / "exp_sam2_finetuned_256", "*-downscaled-mask-nms.shp"),
    "SAM2-auto":       (work_dir / "exp_sam2_auto_256",      "*-mask-nms.shp"),
}

res             = raster_metadata.get_resolution(in_raster)[0]
AREAL_THRESHOLD = (res ** 2) * (4.74 ** 2)
AR_MIN, AR_MAX  = 1.2, 2.0
BINS            = np.linspace(0, 180, 37)

# Canny defaults — tune these if results look empty
CANNY_SIGMA = 1.0
CANNY_LOW   = 0.1
CANNY_HIGH  = 0.2

MAX_N = None  # set e.g. 5000 for a quick test run

OUT_DIR = Path("figures_paper"); OUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 9, "axes.titlesize": 9, "figure.dpi": 150})
print(f"Resolution: {res:.4f} m/px   Areal threshold: {AREAL_THRESHOLD:.4f} m²")

In [ ]:
# Baseline estimators (reproduced from 01b so this notebook is self-contained)

def _ellipse_mrr(poly, seg_res):
    try:
        geom = segmentize(poly, seg_res) if seg_res is not None else poly
        row  = pd.Series({"geometry": geom})
        ellipse_poly, _, _, _ = fitEllipse(row)
        mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
        _, _, long_ax, short_ax, _, _, _, angle180 = boulder_row(mrr_row)
        if short_ax < 1e-6:
            return None
        return angle180, long_ax / short_ax
    except Exception:
        return None


def m_baseline(poly):
    """Current pipeline: segmentize → EllipseModel → MRR."""
    return _ellipse_mrr(poly, res)


def m_moments(poly):
    """Polygon area moments — Green's theorem PCA. Also used as AR filter for raster methods."""
    try:
        coords = np.array(poly.exterior.coords[:-1])
        cx, cy = poly.centroid.x, poly.centroid.y
        x, y   = coords[:, 0] - cx, coords[:, 1] - cy
        xn, yn = np.roll(x, -1), np.roll(y, -1)
        cross  = x * yn - xn * y
        A = 0.5 * np.abs(cross.sum())
        if A < 1e-12:
            return None
        mu20 = np.sum((x**2 + x*xn + xn**2) * cross) / (6 * A)
        mu02 = np.sum((y**2 + y*yn + yn**2) * cross) / (6 * A)
        mu11 = np.sum((x*yn + 2*x*y + 2*xn*yn + xn*y) * cross) / (24 * A)
        cov  = np.array([[mu20, mu11], [mu11, mu02]])
        eigvals, eigvecs = np.linalg.eigh(cov)
        long_vec = eigvecs[:, 1]
        angle180 = np.degrees(np.arctan2(long_vec[0], long_vec[1])) % 180
        ar = np.sqrt(np.abs(eigvals[1]) / max(np.abs(eigvals[0]), 1e-12))
        return angle180, max(ar, 1 / ar if ar > 0 else 1.0)
    except Exception:
        return None

print("Baseline estimators ready.")

In [ ]:
def _prep_image(poly, src):
    """Crop raster to polygon bounding box, normalize by interior pixel range.
    Exterior pixels keep their actual image values — NOT zeroed — so Sobel does not
    see an artificial hard edge at the staircase mask boundary.
    The interior mask is used only to (a) compute normalization stats and
    (b) restrict which pixels contribute to gradient sums.
    Returns (img_float, img_normed, interior_mask) or (None, None, None) if too small.
    """
    out_image, _ = rio_mask(src, [poly], crop=True, nodata=0)
    img = out_image[0].astype(float)
    interior = img != 0
    if interior.sum() < 9:
        return None, None, None
    lo, hi = img[interior].min(), img[interior].max()
    img_normed = (img - lo) / (hi - lo + 1e-12)
    # deliberately NOT zeroing exterior: img_normed[~interior] = 0.0 would create an
    # artificial step at the staircase boundary and dominate the structure tensor
    return img, img_normed, interior


def canny_pca(poly, src, sigma=1.0, low_threshold=0.1, high_threshold=0.2):
    """Canny on image → PCA on edge pixel *coordinates*.
    Long axis = direction of maximum spatial spread of Canny edge pixels.
    AR from eigenvalue ratio of edge pixel covariance.
    """
    try:
        _, img_normed, interior = _prep_image(poly, src)
        if img_normed is None:
            return None
        edges = skimage_canny(img_normed, sigma=sigma,
                              low_threshold=low_threshold,
                              high_threshold=high_threshold,
                              mask=interior)
        if edges.sum() < 4:
            return None
        # (row, col) → use [col, row] = [East, South] for PCA
        edge_pts = np.column_stack(np.where(edges))[:, [1, 0]].astype(float)
        edge_pts -= edge_pts.mean(axis=0)
        cov = np.cov(edge_pts.T)
        eigvals, eigvecs = np.linalg.eigh(cov)    # ascending
        long_vec = eigvecs[:, 1]                   # largest eigenvalue = dominant spread
        ev_east  = long_vec[0]                     # col = East
        ev_south = long_vec[1]                     # row = South
        angle180 = np.degrees(np.arctan2(ev_east, -ev_south)) % 180
        ar = np.sqrt(np.abs(eigvals[1]) / max(np.abs(eigvals[0]), 1e-12))
        return angle180, max(ar, 1 / ar if ar > 0 else 1.0)
    except Exception:
        return None


def canny_structure_tensor(poly, src, sigma=1.0, low_threshold=0.1, high_threshold=0.2):
    """Canny on image → structure tensor at edge pixels only.
    Long axis = eigenvector of smallest eigenvalue (⊥ dominant gradient at edges).
    Sobel computed on full tile (no zero-fill), summed only at Canny edge pixels.
    Returns AR=0 — use moments AR for filter.
    """
    try:
        _, img_normed, interior = _prep_image(poly, src)
        if img_normed is None:
            return None
        edges = skimage_canny(img_normed, sigma=sigma,
                              low_threshold=low_threshold,
                              high_threshold=high_threshold,
                              mask=interior)
        if edges.sum() < 4:
            return None
        Gx = nd_sobel(img_normed, axis=1)   # East gradient — computed on full tile
        Gy = nd_sobel(img_normed, axis=0)   # South gradient
        gx_e, gy_e = Gx[edges], Gy[edges]  # restrict sum to edge pixels
        J = np.array([[np.sum(gx_e**2),      np.sum(gx_e * gy_e)],
                      [np.sum(gx_e * gy_e),  np.sum(gy_e**2)]])
        if np.sum(J) < 1e-12:
            return None
        eigvals, eigvecs = np.linalg.eigh(J)
        long_vec = eigvecs[:, 0]            # smallest eigenvalue = long axis
        ev_east  =  long_vec[0]
        ev_south =  long_vec[1]
        angle180 = np.degrees(np.arctan2(ev_east, -ev_south)) % 180
        return angle180, 0.0               # moments AR used for filter
    except Exception:
        return None


def full_structure_tensor(poly, src):
    """Structure tensor on ALL interior pixels.
    Sobel computed on full tile (no zero-fill), summed only over interior pixels.
    Returns AR=0 — use moments AR for filter.
    """
    try:
        _, img_normed, interior = _prep_image(poly, src)
        if img_normed is None:
            return None
        Gx = nd_sobel(img_normed, axis=1)   # computed on full tile
        Gy = nd_sobel(img_normed, axis=0)
        J = np.array([[np.sum(Gx[interior]**2),          np.sum(Gx[interior]*Gy[interior])],
                      [np.sum(Gx[interior]*Gy[interior]), np.sum(Gy[interior]**2)]])
        if np.sum(J) < 1e-12:
            return None
        eigvals, eigvecs = np.linalg.eigh(J)
        long_vec = eigvecs[:, 0]
        ev_east  =  long_vec[0]
        ev_south =  long_vec[1]
        angle180 = np.degrees(np.arctan2(ev_east, -ev_south)) % 180
        return angle180, 0.0
    except Exception:
        return None

print("Canny estimators ready (no zero-fill fix applied).")
print(f"Default Canny settings: sigma={CANNY_SIGMA}, low={CANNY_LOW}, high={CANNY_HIGH}")

In [ ]:
# Load YOLO detections
pred_dir, glob_pat = PRED_CONFIGS["YOLOv8"]
shp_paths = sorted(pred_dir.glob(glob_pat))
assert shp_paths, f"No shapefiles in {pred_dir}"

gdfs_all = [gpd.read_file(p) for p in shp_paths]
gdf = gpd.GeoDataFrame(pd.concat(gdfs_all, ignore_index=True), crs=gdfs_all[0].crs)
gdf["poly_area"] = gdf.geometry.area
gdf = gdf[gdf["poly_area"] >= AREAL_THRESHOLD].reset_index(drop=True)
gdf["d_px"] = np.sqrt(4 * gdf["poly_area"] / np.pi) / res

if MAX_N is not None and len(gdf) > MAX_N:
    gdf = gdf.sample(MAX_N, random_state=42).reset_index(drop=True)
    print(f"Subsampled to {MAX_N:,}")
print(f"YOLO detections: {len(gdf):,}")

baseline_r = []
moments_r  = []
cpca_r     = []
ctensor_r  = []
ftensor_r  = []

with rasterio.open(in_raster) as src:
    for geom in tqdm(gdf.geometry, desc="Computing orientations"):
        if geom is None or geom.is_empty:
            for lst in [baseline_r, moments_r, cpca_r, ctensor_r, ftensor_r]:
                lst.append(None)
            continue
        baseline_r.append(m_baseline(geom))
        moments_r.append(m_moments(geom))
        cpca_r.append(   canny_pca(geom, src, CANNY_SIGMA, CANNY_LOW, CANNY_HIGH))
        ctensor_r.append(canny_structure_tensor(geom, src, CANNY_SIGMA, CANNY_LOW, CANNY_HIGH))
        ftensor_r.append(full_structure_tensor(geom, src))

def _angle_with_filter(results, use_own_ar, moments_results=None):
    out = []
    for i, r in enumerate(results):
        if r is None:
            out.append(np.nan); continue
        angle, ar = r
        if use_own_ar:
            passes = AR_MIN <= ar <= AR_MAX
        else:
            m = moments_results[i]
            passes = (m is not None) and (AR_MIN <= m[1] <= AR_MAX)
        out.append(angle if passes else np.nan)
    return out

gdf["baseline"]  = _angle_with_filter(baseline_r,  use_own_ar=True)
gdf["moments"]   = _angle_with_filter(moments_r,   use_own_ar=True)
gdf["cpca"]      = _angle_with_filter(cpca_r,      use_own_ar=True)
gdf["ctensor"]   = _angle_with_filter(ctensor_r,   use_own_ar=False, moments_results=moments_r)
gdf["ftensor"]   = _angle_with_filter(ftensor_r,   use_own_ar=False, moments_results=moments_r)

for col in ["baseline", "moments", "cpca", "ctensor", "ftensor"]:
    n = gdf[col].notna().sum()
    print(f"{col:<12}  {n:>7,} elongated boulders")

## Experiment A — All 5 methods × size bins

**What to look for:**
- Do Canny-based methods (cpca, ctensor) show fewer spikes than the baseline? → image gradients are less biased
- Do Canny methods agree with each other? → edge location vs edge gradient direction give same answer
- Do Canny methods agree with the full structure tensor? → restricting to edge pixels doesn't change result
- Do spikes weaken in the large-boulder row? → rasterization is the source

In [ ]:
SIZE_BINS = [
    (0,    8,   "small  (< 8 px)"),
    (8,    20,  "medium (8–20 px)"),
    (20,   1e9, "large  (> 20 px)"),
]

METHOD_COLS = {
    "baseline\n(segmentize→ellipse→MRR)": "baseline",
    "moments\n(area PCA)": "moments",
    "Canny PCA\n(edge locations)": "cpca",
    "Canny tensor\n(edge gradients)": "ctensor",
    "full tensor\n(all interior px)": "ftensor",
}
COLORS = ["#4C72B0", "#8172B3", "#DD8452", "#C44E52", "#55A868"]

fig, axes = plt.subplots(len(SIZE_BINS), len(METHOD_COLS),
                         figsize=(3.0 * len(METHOD_COLS), 2.6 * len(SIZE_BINS)),
                         sharex=True)

for row_i, (dlo, dhi, size_label) in enumerate(SIZE_BINS):
    size_mask = (gdf["d_px"] >= dlo) & (gdf["d_px"] < dhi)
    for col_j, (mname, col) in enumerate(METHOD_COLS.items()):
        ax = axes[row_i][col_j]
        angles = gdf.loc[size_mask, col].dropna().values
        counts, _ = np.histogram(angles, bins=BINS)
        cx = (BINS[:-1] + BINS[1:]) / 2
        D, p = (kstest(angles / 180.0, "uniform") if len(angles) >= 10 else (np.nan, np.nan))
        ax.bar(cx, counts, width=4.5, color=COLORS[col_j], edgecolor="white", lw=0.3)
        ax.axhline(len(angles) / len(cx), color="k", ls="--", lw=0.6, alpha=0.5)
        ax.set(xlim=(0, 180), xticks=[0, 45, 90, 135, 180])
        ax.set_title(f"{mname}\n{size_label}  n={len(angles):,}  D={D:.3f}", fontsize=7)
        ax.spines[["top", "right"]].set_visible(False)
        if col_j == 0:
            ax.set_ylabel("Count")
        if row_i == len(SIZE_BINS) - 1:
            ax.set_xlabel("Orientation (°)")

fig.suptitle(f"YOLO — Canny vs baseline (σ={CANNY_SIGMA}, low={CANNY_LOW}, high={CANNY_HIGH})",
             y=1.01, fontsize=10)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_canny_size_grid.pdf", bbox_inches="tight")
plt.show()
print("Saved fig_canny_size_grid.pdf")

## Experiment B — Canny σ sensitivity

Canny internally applies Gaussian pre-smoothing at scale σ before computing gradients.
Smaller σ → sensitive to fine pixel-level edges (may include staircase edges).
Larger σ → only detects coarser, smoother edges (more robust to staircase).

If spikes weaken with larger σ, it suggests Canny is detecting staircase edges at small σ.

In [ ]:
SIGMA_VARIANTS = [0.5, 1.0, 2.0, 3.0]

sigma_angles = {}
with rasterio.open(in_raster) as src:
    for sigma in SIGMA_VARIANTS:
        angles = []
        for geom in tqdm(gdf.geometry, desc=f"Canny PCA σ={sigma}", leave=False):
            if geom is None or geom.is_empty:
                angles.append(np.nan); continue
            r = canny_pca(geom, src, sigma=sigma,
                          low_threshold=CANNY_LOW, high_threshold=CANNY_HIGH)
            angles.append(r[0] if (r and AR_MIN <= r[1] <= AR_MAX) else np.nan)
        sigma_angles[sigma] = angles
        n = sum(1 for a in angles if not np.isnan(a))
        print(f"  σ={sigma}: {n:,} elongated boulders")

fig, axes = plt.subplots(1, len(SIGMA_VARIANTS),
                         figsize=(3.0 * len(SIGMA_VARIANTS), 2.8), sharex=True)
for ax, sigma in zip(axes, SIGMA_VARIANTS):
    angles = np.array([a for a in sigma_angles[sigma] if not np.isnan(a)])
    counts, _ = np.histogram(angles, bins=BINS)
    cx = (BINS[:-1] + BINS[1:]) / 2
    D, p = (kstest(angles / 180.0, "uniform") if len(angles) >= 10 else (np.nan, np.nan))
    ax.bar(cx, counts, width=4.5, color="#DD8452", edgecolor="white", lw=0.3)
    ax.axhline(len(angles) / len(cx), color="k", ls="--", lw=0.6, alpha=0.5)
    ax.set_title(f"Canny PCA σ={sigma}\nn={len(angles):,}  D={D:.3f}", fontsize=8)
    ax.set(xlim=(0, 180), xticks=[0, 45, 90, 135, 180], xlabel="Orientation (°)")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Count")
fig.suptitle(f"Canny PCA — σ sensitivity (low={CANNY_LOW}, high={CANNY_HIGH})",
             y=1.02, fontsize=10)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_canny_sigma_sensitivity.pdf", bbox_inches="tight")
plt.show()
print("Saved fig_canny_sigma_sensitivity.pdf")

## KS Statistics — Uniformity test

D-statistic: distance from a uniform distribution over [0°, 180°]. Higher = more biased.

In [ ]:
rows = []
for dlo, dhi, size_label in SIZE_BINS:
    size_mask = (gdf["d_px"] >= dlo) & (gdf["d_px"] < dhi)
    for mname, col in METHOD_COLS.items():
        angles = gdf.loc[size_mask, col].dropna().values
        if len(angles) >= 10:
            D, p = kstest(angles / 180.0, "uniform")
        else:
            D, p = np.nan, np.nan
        rows.append({"Method": mname.replace("\n", " "),
                     "Size bin": size_label, "n": len(angles),
                     "D": round(D, 3) if not np.isnan(D) else np.nan,
                     "p": f"{p:.2e}" if not np.isnan(p) else "—"})

ks_df = pd.DataFrame(rows)
print(ks_df.to_string(index=False))